In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
trade_df = pd.read_csv("/Users/navikamaglani/Documents/personal/supply_chain_disruption_ai/data/raw/us_trade_data.csv")

trade_df.head()

,CTY_CODE,CTY_NAME,GEN_VAL_MO,time
0,-,TOTAL FOR ALL COUNTRIES,195796204072,2020-01-01
1,0003,EUROPEAN UNION,35071255143,2020-01-01
2,0014,PACIFIC RIM COUNTRIES,65080414292,2020-01-01
3,0017,CAFTA-DR,1786453267,2020-01-01
4,0020,USMCA (NAFTA),53489130119,2020-01-01


In [5]:
trade_df["GEN_VAL_MO"] = pd.to_numeric(trade_df["GEN_VAL_MO"], errors="coerce")
trade_df["time"] = pd.to_datetime(trade_df["time"])

trade_df = trade_df.sort_values(["CTY_CODE", "time"])

trade_df.head()

,CTY_CODE,CTY_NAME,GEN_VAL_MO,time
0,-,TOTAL FOR ALL COUNTRIES,195796204072,2020-01-01
249,-,TOTAL FOR ALL COUNTRIES,178107959988,2020-02-01
500,-,TOTAL FOR ALL COUNTRIES,193922847899,2020-03-01
752,-,TOTAL FOR ALL COUNTRIES,165611885689,2020-04-01
1005,-,TOTAL FOR ALL COUNTRIES,163494079397,2020-05-01


In [6]:
trade_df["year"] = trade_df["time"].dt.year
trade_df["month"] = trade_df["time"].dt.month
trade_df["quarter"] = trade_df["time"].dt.quarter

In [7]:
trade_df["mom_change"] = trade_df.groupby("CTY_CODE")["GEN_VAL_MO"].pct_change()

trade_df["rolling_3_month_avg"] = (
    trade_df.groupby("CTY_CODE")["GEN_VAL_MO"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

trade_df["rolling_6_month_avg"] = (
    trade_df.groupby("CTY_CODE")["GEN_VAL_MO"]
    .transform(lambda x: x.rolling(window=6, min_periods=1).mean())
)

trade_df.head()

,CTY_CODE,CTY_NAME,GEN_VAL_MO,time,year,month,quarter,mom_change,rolling_3_month_avg,rolling_6_month_avg
0,-,TOTAL FOR ALL COUNTRIES,195796204072,2020-01-01,2020,1,1,NaN,1.957962e+11,1.957962e+11
249,-,TOTAL FOR ALL COUNTRIES,178107959988,2020-02-01,2020,2,1,-0.090340,1.869521e+11,1.869521e+11
500,-,TOTAL FOR ALL COUNTRIES,193922847899,2020-03-01,2020,3,1,0.088794,1.892757e+11,1.892757e+11
752,-,TOTAL FOR ALL COUNTRIES,165611885689,2020-04-01,2020,4,2,-0.145991,1.792142e+11,1.833597e+11
1005,-,TOTAL FOR ALL COUNTRIES,163494079397,2020-05-01,2020,5,2,-0.012788,1.743429e+11,1.793866e+11


In [8]:
trade_df["historical_mean"] = (
    trade_df.groupby("CTY_CODE")["GEN_VAL_MO"]
    .transform(lambda x: x.expanding().mean())
)

trade_df["historical_std"] = (
    trade_df.groupby("CTY_CODE")["GEN_VAL_MO"]
    .transform(lambda x: x.expanding().std())
)

trade_df["deviation_from_avg"] = (
    (trade_df["GEN_VAL_MO"] - trade_df["historical_mean"]) /
    trade_df["historical_std"]
)

trade_df["high_disruption_risk"] = np.where(
    (trade_df["mom_change"] < -0.20) |
    (trade_df["deviation_from_avg"] < -1.5),
    1,
    0
)

trade_df.head()

,CTY_CODE,CTY_NAME,GEN_VAL_MO,time,year,month,quarter,mom_change,rolling_3_month_avg,rolling_6_month_avg,historical_mean,historical_std,deviation_from_avg,high_disruption_risk
0,-,TOTAL FOR ALL COUNTRIES,195796204072,2020-01-01,2020,1,1,NaN,1.957962e+11,1.957962e+11,1.957962e+11,NaN,NaN,0
249,-,TOTAL FOR ALL COUNTRIES,178107959988,2020-02-01,2020,2,1,-0.090340,1.869521e+11,1.869521e+11,1.869521e+11,1.250748e+10,-0.707107,0
500,-,TOTAL FOR ALL COUNTRIES,193922847899,2020-03-01,2020,3,1,0.088794,1.892757e+11,1.892757e+11,1.892757e+11,9.716773e+09,0.478263,0
752,-,TOTAL FOR ALL COUNTRIES,165611885689,2020-04-01,2020,4,2,-0.145991,1.792142e+11,1.833597e+11,1.833597e+11,1.424561e+10,-1.245846,0
1005,-,TOTAL FOR ALL COUNTRIES,163494079397,2020-05-01,2020,5,2,-0.012788,1.743429e+11,1.793866e+11,1.793866e+11,1.520302e+10,-1.045352,0


In [9]:
trade_df["high_disruption_risk"].value_counts(normalize=True)

high_disruption_risk
0    0.740861
1    0.259139
Name: proportion, dtype: float64

In [10]:
os.makedirs("/Users/navikamaglani/Documents/personal/supply_chain_disruption_ai/data/processed", exist_ok=True)

trade_df.to_csv(
    "/Users/navikamaglani/Documents/personal/supply_chain_disruption_ai/data/processed/trade_features.csv",
    index=False
)

print("Feature engineering dataset saved!")

Feature engineering dataset saved!
